In [1]:
import sys

sys.path.append("..")


import dotenv

import wandb
from src.api.run.xgboost import sweep_xgboost
from src.api.sweep import wandb_sweep
from src.models.xgboost.submission import create_submission

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: v8-luky (aicomp-mmlm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
max_runs = 100
sweep_config = {
    "name": "XGBoost",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "xgboost_config": {
            "parameters": {
                "device": {"value": "cpu"},
                "objective": {"values": ["reg:squarederror", "reg:logistic", "binary:logistic", "binary:hinge"]},
                "booster": {"values": ["gbtree", "gblinear", "dart"]},
                "learning_rate": {"distribution": "uniform", "min": 0.001, "max": 1.0},
                "gamma": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "max_depth": {"distribution": "int_uniform", "min": 3, "max": 15},
                "min_child_weight": {"values": [0.0, 1.0, 2.0, 3.0]},
                "subsample": {"values": [1.0, 0.9, 0.8, 0.7]},
                "colsample_bytree": {"values": [1.0, 0.9, 0.8, 0.7]},
                "colsample_bylevel": {"values": [1.0, 0.9, 0.8, 0.7]},
                "colsample_bynode": {"values": [1.0, 0.9, 0.8, 0.7]},
                "reg_lambda": {"distribution": "uniform", "min": 0.0, "max": 2.0},
                "reg_alpha": {"distribution": "uniform", "min": 0.0, "max": 2.0},
                "max_bin": {"distribution": "int_uniform", "min": 16, "max": 512},
                "grow_policy": {"values": ["depthwise", "lossguide"]},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_xgboost, run_count=max_runs, project="xgboost-regressor")

In [ ]:
max_runs = 40
sweep_config = {
    "name": "XGBoost: Finetune",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "xgboost_config": {
            "parameters": {
                "device": {"value": "cpu"},
                "booster": {"values": ["gbtree", "gblinear"]},
                "learning_rate": {"distribution": "uniform", "min": 0.001, "max": 0.3},
                "gamma": {"distribution": "uniform", "min": 0.0, "max": 0.2},
                "max_depth": {"distribution": "int_uniform", "min": 5, "max": 40},
                "colsample_bytree": {"values": [0.8, 0.7, 0.6]},
                "colsample_bylevel": {"values": [1.0, 0.8, 0.6]},
                "reg_lambda": {"distribution": "uniform", "min": 1.0, "max": 3.0},
                "reg_alpha": {"distribution": "uniform", "min": 0.0, "max": 1.0},
                "max_bin": {"distribution": "int_uniform", "min": 64, "max": 256},
                "grow_policy": {"value": "lossguide"},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 8, "max": 85},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_xgboost, run_count=max_runs, project="xgboost-regressor")

### Submission from Best Model

In [5]:
create_submission(season=2025, run_id="i2klbfcj", entity="aicomp-mmlm", project="xgboost-regressor")

Config: {'run_config': {'num_features': 71, 'start_season': 2003, 'valid_season': 2024}, 'xgboost_config': {'gamma': 0.02042301213967479, 'device': 'cpu', 'booster': 'gblinear', 'max_bin': 141, 'max_depth': 6, 'objective': 'reg:logistic', 'reg_alpha': 0.07283077060601983, 'subsample': 1, 'reg_lambda': 1.4590567759677, 'grow_policy': 'depthwise', 'learning_rate': 0.27652708772305523, 'colsample_bynode': 0.9, 'colsample_bytree': 0.7, 'min_child_weight': 0, 'colsample_bylevel': 0.7}}
Run Config: RunConfig(num_features=71, valid_season=2024, start_season=2003, data_loader='season_average', data_loader_config=None)
Hyperparameters: XGBHyperparamConfig(num_rounds=400, device='cpu', objective='reg:logistic', booster='gblinear', learning_rate=0.27652708772305523, gamma=0.02042301213967479, max_depth=6, min_child_weight=0, num_parallel_tree=1, max_delta_step=0, subsample=1, colsample_bytree=0.7, colsample_bylevel=0.7, colsample_bynode=0.9, reg_lambda=1.4590567759677, reg_alpha=0.072830770606019

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [20:42:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "colsample_bylevel", "colsample_bynode", "colsample_bytree", "gamma", "grow_policy", "max_bin", "max_delta_step", "max_depth", "min_child_weight", "num_parallel_tree", "subsample", "tree_method" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


metrics: {'train_brier': np.float64(0.16749845189526694)}, step: None

Created submission at: C:\Users\kybur\Repos\HSLU\aicomp\code\submissions\submission_xgboost_2025.csv
Created submission at: C:\Users\kybur\Repos\HSLU\aicomp\code\submissions\submission_xgboost_2025.csv


WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_xgboost_2025.csv')

### Sweep on default features

In [3]:
max_runs = 100
sweep_config = {
    "name": "XGBoost: Default Features",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "xgboost_config": {
            "parameters": {
                "device": {"value": "cpu"},
                "learning_rate": {"distribution": "uniform", "min": 0.001, "max": 0.5},
                "gamma": {"distribution": "uniform", "min": 0.0, "max": 0.3},
                "max_depth": {"distribution": "int_uniform", "min": 3, "max": 20},
                "subsample": {"values": [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]},
                "colsample_bytree": {"values": [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]},
                "colsample_bylevel": {"values": [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]},
                "colsample_bynode": {"values": [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]},
                "reg_lambda": {"distribution": "uniform", "min": 0.5, "max": 3.0},
                "reg_alpha": {"distribution": "uniform", "min": 0.0, "max": 1.5},
                "max_bin": {"distribution": "int_uniform", "min": 8, "max": 256},
                "grow_policy": {"values": ["lossguide"]},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"value": 0},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_xgboost, run_count=max_runs, project="xgboost-regressor")

### Submission from best Model

In [3]:
create_submission(
    season=2025,
    run_id="13xu0ucm",
    submission_affix="default_features",
    entity="aicomp-mmlm",
    project="xgboost-regressor",
)

Config: {'run_config': {'num_features': 0, 'start_season': 2003, 'valid_season': 2024}, 'xgboost_config': {'gamma': 0.20265956205717503, 'device': 'cpu', 'max_bin': 256, 'max_depth': 3, 'reg_alpha': 1.10577664426484, 'subsample': 0.9, 'reg_lambda': 2.053682229915379, 'grow_policy': 'lossguide', 'learning_rate': 0.15263869650898068, 'colsample_bynode': 0.5, 'colsample_bytree': 0.6, 'colsample_bylevel': 1}}
Run Config: RunConfig(num_features=0, valid_season=2024, start_season=2003, data_loader='season_average', data_loader_config=None)
Hyperparameters: XGBHyperparamConfig(num_rounds=400, device='cpu', objective='reg:squarederror', booster='gbtree', learning_rate=0.15263869650898068, gamma=0.20265956205717503, max_depth=3, min_child_weight=1.0, num_parallel_tree=1, max_delta_step=0, subsample=0.9, colsample_bytree=0.6, colsample_bylevel=1, colsample_bynode=0.5, reg_lambda=2.053682229915379, reg_alpha=1.10577664426484, tree_method='hist', max_bin=256, grow_policy='lossguide', seed=42)
 {'r

WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_xgboost_default_features_2025.csv')

## Sweep with Weighted Season Average DataLoader

In [6]:
max_runs = 100
sweep_config = {
    "name": "XGBoost",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "xgboost_config": {
            "parameters": {
                "device": {"value": "cpu"},
                "objective": {"values": ["reg:squarederror", "reg:logistic", "binary:logistic", "binary:hinge"]},
                "booster": {"values": ["gbtree", "gblinear", "dart"]},
                "learning_rate": {"distribution": "uniform", "min": 0.001, "max": 1.0},
                "gamma": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "max_depth": {"distribution": "int_uniform", "min": 3, "max": 15},
                "min_child_weight": {"values": [0.0, 1.0, 2.0, 3.0]},
                "subsample": {"values": [1.0, 0.9, 0.8, 0.7]},
                "colsample_bytree": {"values": [1.0, 0.9, 0.8, 0.7]},
                "colsample_bylevel": {"values": [1.0, 0.9, 0.8, 0.7]},
                "colsample_bynode": {"values": [1.0, 0.9, 0.8, 0.7]},
                "reg_lambda": {"distribution": "uniform", "min": 0.0, "max": 2.0},
                "reg_alpha": {"distribution": "uniform", "min": 0.0, "max": 2.0},
                "max_bin": {"distribution": "int_uniform", "min": 16, "max": 512},
                "grow_policy": {"values": ["depthwise", "lossguide"]},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2000},
                "start_season": {"value": 2000},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
                "data_loader": {"value": "weighted_season_average"},
                "data_loader_config": {
                    "parameters": {
                        "regular_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "tourney_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "discount_factor": {"distribution": "uniform", "min": 0.9, "max": 1.0},
                    }
                },
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_xgboost, run_count=max_runs, project="xgboost-regressor")

### Submission from best Model

In [4]:
create_submission(
    season=2025, run_id="0ztpg0gd", submission_affix="weighted_avg", entity="aicomp-mmlm", project="xgboost-regressor"
)

Config: {'run_config': {'data_loader': 'weighted_season_average', 'num_features': 19, 'start_season': 2000, 'valid_season': 2000, 'data_loader_config': {'regular_weight': 0.5422587700615304, 'tourney_weight': 0.6182492888693102, 'discount_factor': 0.9919287156610832}}, 'xgboost_config': {'gamma': 0.3704736561877338, 'device': 'cpu', 'booster': 'gbtree', 'max_bin': 256, 'max_depth': 6, 'objective': 'reg:squarederror', 'reg_alpha': 0.1454475184670343, 'subsample': 1, 'reg_lambda': 1.6713082209772132, 'grow_policy': 'lossguide', 'learning_rate': 0.09231373173680534, 'colsample_bynode': 0.9, 'colsample_bytree': 1, 'min_child_weight': 1, 'colsample_bylevel': 0.8}}
Run Config: RunConfig(num_features=19, valid_season=2000, start_season=2000, data_loader='weighted_season_average', data_loader_config={'regular_weight': 0.5422587700615304, 'tourney_weight': 0.6182492888693102, 'discount_factor': 0.9919287156610832})
Hyperparameters: XGBHyperparamConfig(num_rounds=400, device='cpu', objective='re

WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_xgboost_weighted_avg_2025.csv')